# QLoRA Fine-Tuning on NRP JupyterHub

This notebook fine-tunes **Qwen2.5 3B** on physics paper abstracts using **QLoRA** — all running on a free GPU from the National Research Platform.

**What is QLoRA?**
1. Load the base model in 4-bit precision (~1.5 GB instead of ~6 GB)
2. Attach tiny trainable "adapter" matrices (LoRA) to each layer
3. Train only those adapters — everything else stays frozen

The result: a small adapter file (~20-50 MB) that makes the model write like a domain expert.

**Prerequisites:**
- Launch this notebook from [NRP JupyterHub](https://jupyterhub-west.nrp-nautilus.io) with a **GPU server profile**
- Upload `train.jsonl` and `val.jsonl` to the `data/` folder (or run the data prep scripts first)
- (Optional) A HuggingFace token speeds up downloads: https://huggingface.co/settings/tokens

## 1. Environment setup

In [ ]:
!pip install -q torch transformers datasets accelerate peft trl bitsandbytes huggingface_hub

In [ ]:
# Verify GPU is available
!nvidia-smi

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    # Auto-detect bf16 support (A100, H200, etc.)
    print(f"BF16 support: {torch.cuda.is_bf16_supported()}")

## 2. Configuration

All the knobs in one place. Adjust `BATCH_SIZE` down to 2 or 1 if you run out of VRAM.

In [ ]:
import os

# --- Cache location ---
# NRP JupyterHub home dirs are only 5 GB — model weights won't fit.
# Download to /tmp instead (backed by overlay, ~170 GB free).
os.environ["HF_HOME"] = "/tmp/hf_cache"

# --- Model ---
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

# --- Data ---
TRAIN_DATA = "./data/train.jsonl"
VAL_DATA = "./data/val.jsonl"

# --- Training ---
OUTPUT_DIR = "./output/physics-adapter"
EPOCHS = 3
BATCH_SIZE = 4          # Reduce to 2 or 1 if you get OOM errors
GRAD_ACCUM_STEPS = 4    # Effective batch size = BATCH_SIZE * GRAD_ACCUM_STEPS
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH = 512

# --- LoRA ---
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# --- Precision ---
# Use bf16 on A100/H200, fp16 on older GPUs (T4, V100)
USE_BF16 = torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False
print(f"Using {'bf16' if USE_BF16 else 'fp16'} mixed precision")

## 3. (Optional) HuggingFace authentication

Qwen2.5 is an **ungated** model — no license approval needed! Logging in is optional but helps avoid download rate limits.

In [ ]:
from huggingface_hub import login

# Optional: speeds up downloads and avoids rate limits
# login(token="hf_your_token_here")

# Or use interactive prompt:
# login()

## 4. Load the base model in 4-bit precision

This is the "Q" in QLoRA — we load the 3B-parameter model in 4-bit NormalFloat (NF4) precision, shrinking it from ~6 GB to ~1.5 GB in memory.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
)

# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model
print("Loading base model in 4-bit precision (this may take a minute)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print(f"Model loaded on: {model.device}")

## 5. Attach LoRA adapters

This is the "LoRA" in QLoRA — we freeze the entire base model and inject small trainable matrices into the attention and MLP layers. Instead of training 3 billion parameters, we train ~27 million (0.84%).

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare the quantized model for training
model = prepare_model_for_kbit_training(model)

# LoRA configuration
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # attention
        "gate_proj", "up_proj", "down_proj",       # MLP
    ],
)

model = get_peft_model(model, lora_config)

# How few parameters are we actually training?
model.print_trainable_parameters()

## 6. Load training data

Our training data is ~1,800 arXiv abstracts from condensed matter physics, formatted as chat conversations.

In [ ]:
from datasets import load_dataset

dataset_train = load_dataset("json", data_files=TRAIN_DATA, split="train")
dataset_val = load_dataset("json", data_files=VAL_DATA, split="train")

print(f"Train examples: {len(dataset_train)}")
print(f"Val examples:   {len(dataset_val)}")
print()
print("Sample training example (first 300 chars):")
print(dataset_train[0]["text"][:300])

## 7. Set up the trainer

We use `SFTTrainer` from the `trl` library — it handles tokenization and formatting for us.

In [ ]:
from trl import SFTConfig, SFTTrainer

training_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.1,
    fp16=not USE_BF16,
    bf16=USE_BF16,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,           # Keep only the 2 best checkpoints
    load_best_model_at_end=True,
    report_to="none",
    optim="paged_adamw_8bit",
    max_length=MAX_SEQ_LENGTH,
)

trainer = SFTTrainer(
    model=model,
    args=training_config,
    train_dataset=dataset_train,
    eval_dataset=dataset_val,
    processing_class=tokenizer,
)

print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"Training steps per epoch: {len(dataset_train) // (BATCH_SIZE * GRAD_ACCUM_STEPS)}")
print(f"Total training steps: {len(dataset_train) // (BATCH_SIZE * GRAD_ACCUM_STEPS) * EPOCHS}")

## 8. Train!

This will take approximately 30-60 minutes on a T4, or 15-30 minutes on an A100. You'll see a progress bar with loss values.

In [ ]:
trainer.train()

## 9. Save the adapter

We save **only the LoRA adapter** (~20-50 MB), not the full 3B model. Anyone who has the base model can load our adapter on top.

In [ ]:
import os

print("Saving LoRA adapter...")
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Report adapter size
adapter_size_bytes = sum(
    os.path.getsize(os.path.join(OUTPUT_DIR, f))
    for f in os.listdir(OUTPUT_DIR)
    if f.endswith((".safetensors", ".bin"))
)
adapter_size_mb = adapter_size_bytes / (1024 * 1024)

print(f"\nAdapter saved to: {OUTPUT_DIR}")
print(f"Adapter size:     {adapter_size_mb:.1f} MB")
print(f"Base model size:  ~6,000 MB (fp16)")
print(f"Compression:      ~{6000 / max(adapter_size_mb, 0.1):.0f}x smaller")

## 10. Test the fine-tuned model

Let's generate a sample abstract and compare it with what the base model would produce.

In [ ]:
# Generate with the fine-tuned model
prompt = (
    "<|im_start|>system\nYou are an expert academic writer specializing in "
    "condensed matter physics.<|im_end|>\n"
    "<|im_start|>user\nWrite an abstract for a paper titled: "
    "'Topological Phase Transitions in Monolayer Jacutingaite (Pt2HgSe3): "
    "A Kane-Mele Analysis with Spin-Orbit and Rashba Corrections'<|im_end|>\n"
    "<|im_start|>assistant\n"
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
    )

generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
# Extract only the assistant's response
if "assistant\n" in generated:
    generated = generated.split("assistant\n")[-1].strip()

print("=" * 60)
print("FINE-TUNED MODEL OUTPUT")
print("=" * 60)
print(generated)

## 11. (Optional) Upload to HuggingFace

Share your adapter so collaborators can use it with two commands.

In [ ]:
# Uncomment and fill in your details to upload

# from huggingface_hub import HfApi, create_repo
#
# REPO_NAME = "your-username/physics-abstract-writer"  # <-- Change this
#
# create_repo(repo_id=REPO_NAME, repo_type="model", private=True, exist_ok=True)
#
# api = HfApi()
# api.upload_folder(
#     folder_path=OUTPUT_DIR,
#     repo_id=REPO_NAME,
#     repo_type="model",
# )
#
# print(f"Uploaded to: https://huggingface.co/{REPO_NAME}")